<a href="https://colab.research.google.com/github/rudalshan0412-code/attention-is-all-you-need-pytorch/blob/main/05)_Encoder_Layer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Google Drive 연결

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 프로젝트 경로 설정

from pathlib import Path
import sys

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/attention_is_all_you_need"
)

SRC_DIR = PROJECT_ROOT / "src"

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
SRC_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_DIR:", SRC_DIR)

PROJECT_ROOT: /content/drive/MyDrive/attention_is_all_you_need
SRC_DIR: /content/drive/MyDrive/attention_is_all_you_need/src


In [ ]:
# 기존 파일 구조 확인

for file_path in sorted(SRC_DIR.glob("*.py")):
    print(file_path.name)

attention.py
feed_forward.py
multi_head_attention.py
positional_encoding.py


In [ ]:
'''
 Encoder는 Encoder Layer로 이루어져있다(논문의 경우 6개로 구성되어 있다.).
 이때, Encoder Layer를 겹침으로서 Encoder는 더욱 고도화된 정보 표현을 할 수 있다.(일종의 다층 퍼셉트론과 비슷한 흐름이다.)
 '''

'\n Encoder는 Encoder Layer로 이루어져있다(논문의 경우 6개로 구성되어 있다.).\n 이때, Encoder Layer를 겹침으로서 Encoder는 더욱 고도화된 정보 표현을 할 수 있다.(일종의 다층 퍼셉트론과 비슷한 흐름이다.)\n '

In [ ]:
# Residual Connection 먼저 확인

import torch

batch_size = 2
seq_len = 4
d_model = 8

x = torch.randn(
    batch_size,
    seq_len,
    d_model,
)

attention_output = torch.randn(
    batch_size,
    seq_len,
    d_model,
)

residual = x + attention_output

print("x:", x.shape)
print("attention_output:", attention_output.shape)
print("residual:", residual.shape)

x: torch.Size([2, 4, 8])
attention_output: torch.Size([2, 4, 8])
residual: torch.Size([2, 4, 8])


In [ ]:
# nn.Layernorm(d_model) 직접 확인

import torch
import torch.nn as nn

x = torch.tensor(
    [
        [
            [1.0, 2.0, 3.0, 4.0],
            [10.0, 20.0, 30.0, 40.0],
        ]
    ]
) # (1, 2, 4)

print("x shape:", x.shape)

layer_norm = nn.LayerNorm(4) # 평균 0, 표준편차 1로 만들어줌과 동시에 linear의 역할 수행(이때, () 내의 숫자와 들어오는 객체의 마지막 차원 수가 동일해야함)

output = layer_norm(x)

print("\nLayerNorm output:")
print(output)

print("\n각 token의 평균:")
print(output.mean(dim=-1))

print("\n각 token의 분산:")
print(output.var(dim=-1, unbiased=False))

x shape: torch.Size([1, 2, 4])

LayerNorm output:
tensor([[[-1.3416, -0.4472,  0.4472,  1.3416],
         [-1.3416, -0.4472,  0.4472,  1.3416]]],
       grad_fn=<NativeLayerNormBackward0>)

각 token의 평균:
tensor([[0., 0.]], grad_fn=<MeanBackward1>)

각 token의 분산:
tensor([[1.0000, 1.0000]], grad_fn=<VarBackward0>)


In [ ]:
# EncoderLayer 전체 코드

import torch.nn as nn

from src.multi_head_attention import MultiHeadAttention
from src.feed_forward import PositionwiseFeedForward


class EncoderLayer(nn.Module):
    def __init__(
        self,
        d_model,
        num_heads,
        d_ff,
        dropout=0.1,
    ):
        super().__init__()

        self.self_attention = MultiHeadAttention(
            d_model,
            num_heads,
        )

        self.feed_forward = PositionwiseFeedForward(
            d_model,
            d_ff,
        )

        self.dropout1 = nn.Dropout(dropout) # 과적합 방지용으로 랜덤하게 노드를 끔
        self.dropout2 = nn.Dropout(dropout)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        # 1. Multi-Head Self-Attention
        attention_output, attention_weights = self.self_attention( # qkv 이용해서 계산
            x,
            x,
            x,
            mask,
        )

        # 2. 첫 번째 Add & Norm
        attention_output = self.dropout1(attention_output)

        x = self.norm1( # 잔차 연결
            x + attention_output
        )

        # 3. Position-wise Feed Forward Network
        ffn_output = self.feed_forward(x) # 독립적으로 계산

        # 4. 두 번째 Add & Norm
        ffn_output = self.dropout2(ffn_output)

        x = self.norm2(
            x + ffn_output
        )

        return x, attention_weights

In [ ]:
# src/encoder_layer.py 저장

%%writefile /content/drive/MyDrive/attention_is_all_you_need/src/encoder_layer.py

import torch.nn as nn

from src.multi_head_attention import MultiHeadAttention
from src.feed_forward import PositionwiseFeedForward


class EncoderLayer(nn.Module):
    def __init__(
        self,
        d_model,
        num_heads,
        d_ff,
        dropout=0.1,
    ):
        super().__init__()

        self.self_attention = MultiHeadAttention(
            d_model,
            num_heads,
        )

        self.feed_forward = PositionwiseFeedForward(
            d_model,
            d_ff,
        )

        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        # 1. Multi-Head Self-Attention
        attention_output, attention_weights = self.self_attention(
            x,
            x,
            x,
            mask,
        )

        # 2. 첫 번째 Add & Norm
        attention_output = self.dropout1(attention_output)

        x = self.norm1(
            x + attention_output
        )

        # 3. Position-wise Feed Forward Network
        ffn_output = self.feed_forward(x)

        # 4. 두 번째 Add & Norm
        ffn_output = self.dropout2(ffn_output)

        x = self.norm2(
            x + ffn_output
        )

        return x, attention_weights

Writing /content/drive/MyDrive/attention_is_all_you_need/src/encoder_layer.py


In [ ]:
# 저장된 EncoredLayer import

from src.encoder_layer import EncoderLayer

print("EncoderLayer import 성공")

EncoderLayer import 성공


In [ ]:
# 기본 shape 테스트

import torch

torch.manual_seed(42)

batch_size = 2
seq_len = 4

d_model = 8
num_heads = 2
d_ff = 32

dropout = 0.0

encoder_layer = EncoderLayer(
    d_model=d_model,
    num_heads=num_heads,
    d_ff=d_ff,
    dropout=dropout,
)

x = torch.randn(
    batch_size,
    seq_len,
    d_model,
)

output, attention_weights = encoder_layer(
    x,
    mask=None,
)

print("input shape:")
print(x.shape)

print("\noutput shape:")
print(output.shape)

print("\nattention_weights shape:")
print(attention_weights.shape)

input shape:
torch.Size([2, 4, 8])

output shape:
torch.Size([2, 4, 8])

attention_weights shape:
torch.Size([2, 2, 4, 4])


In [ ]:
# Encoder 내부 shape 따라가기

# Encoder Layer 입력
manual_x = x

print("1. 입력 x")
print(manual_x.shape)


# Self-Attention
attention_output, manual_attention_weights = (
    encoder_layer.self_attention(
        manual_x,
        manual_x,
        manual_x,
        None,
    )
)

print("\n2. Self-Attention output")
print(attention_output.shape)

print("\n3. attention_weights")
print(manual_attention_weights.shape)


# Attention Dropout
attention_output = encoder_layer.dropout1(
    attention_output
)

print("\n4. Attention Dropout 이후")
print(attention_output.shape)


# 첫 번째 Residual
first_residual = manual_x + attention_output

print("\n5. 첫 번째 Residual 이후")
print(first_residual.shape)


# 첫 번째 LayerNorm
manual_x = encoder_layer.norm1(
    first_residual
)

print("\n6. 첫 번째 LayerNorm 이후")
print(manual_x.shape)


# FFN
ffn_output = encoder_layer.feed_forward(
    manual_x
)

print("\n7. FFN output")
print(ffn_output.shape)


# FFN Dropout
ffn_output = encoder_layer.dropout2(
    ffn_output
)

print("\n8. FFN Dropout 이후")
print(ffn_output.shape)


# 두 번째 Residual
second_residual = manual_x + ffn_output

print("\n9. 두 번째 Residual 이후")
print(second_residual.shape)


# 두 번째 LayerNorm
manual_output = encoder_layer.norm2(
    second_residual
)

print("\n10. 두 번째 LayerNorm 이후")
print(manual_output.shape)

1. 입력 x
torch.Size([2, 4, 8])

2. Self-Attention output
torch.Size([2, 4, 8])

3. attention_weights
torch.Size([2, 2, 4, 4])

4. Attention Dropout 이후
torch.Size([2, 4, 8])

5. 첫 번째 Residual 이후
torch.Size([2, 4, 8])

6. 첫 번째 LayerNorm 이후
torch.Size([2, 4, 8])

7. FFN output
torch.Size([2, 4, 8])

8. FFN Dropout 이후
torch.Size([2, 4, 8])

9. 두 번째 Residual 이후
torch.Size([2, 4, 8])

10. 두 번째 LayerNorm 이후
torch.Size([2, 4, 8])


In [ ]:
# batch size/ sequence 길이 변경 테스트

test_shapes = [
    (1, 3),
    (2, 4),
    (4, 7),
    (3, 10),
]

for batch_size, seq_len in test_shapes:
    x_test = torch.randn(
        batch_size,
        seq_len,
        d_model,
    )

    output_test, weights_test = encoder_layer(
        x_test,
        mask=None,
    )

    print(
        f"B={batch_size}, S={seq_len} | "
        f"output={tuple(output_test.shape)} | "
        f"attention={tuple(weights_test.shape)}"
    )

B=1, S=3 | output=(1, 3, 8) | attention=(1, 2, 3, 3)
B=2, S=4 | output=(2, 4, 8) | attention=(2, 2, 4, 4)
B=4, S=7 | output=(4, 7, 8) | attention=(4, 2, 7, 7)
B=3, S=10 | output=(3, 10, 8) | attention=(3, 2, 10, 10)


In [ ]:
# Mask = None 테스트

x_test = torch.randn(
    2,
    5,
    d_model,
)

output_test, weights_test = encoder_layer(
    x_test,
    mask=None,
)

print("output:", output_test.shape)
print("attention_weights:", weights_test.shape)

output: torch.Size([2, 5, 8])
attention_weights: torch.Size([2, 2, 5, 5])


In [ ]:
# forward()와 수동 계산 결과 비교

# 현재 dropout = 0.0이기에 dropout의 랜덤성은 없는 상태이다.

forward_output, forward_weights = encoder_layer(
    x,
    mask=None,
)

manual_x = x

manual_attention_output, manual_weights = (
    encoder_layer.self_attention(
        manual_x,
        manual_x,
        manual_x,
        None,
    )
)

manual_attention_output = encoder_layer.dropout1(
    manual_attention_output
)

manual_x = encoder_layer.norm1(
    manual_x + manual_attention_output
)

manual_ffn_output = encoder_layer.feed_forward(
    manual_x
)

manual_ffn_output = encoder_layer.dropout2(
    manual_ffn_output
)

manual_output = encoder_layer.norm2(
    manual_x + manual_ffn_output
)

print(
    "EncoderLayer output 동일:",
    torch.allclose(
        forward_output,
        manual_output,
    )
)

print(
    "Attention weights 동일:",
    torch.allclose(
        forward_weights,
        manual_weights,
    )
)

# True가 나와야함(수동 흐름과 실제 forward 구현이 일치한다는 뜻)

EncoderLayer output 동일: True
Attention weights 동일: True


In [ ]:
# 최종 통합 테스트

import torch

from src.encoder_layer import EncoderLayer


torch.manual_seed(42)

d_model = 8
num_heads = 2
d_ff = 32

encoder_layer = EncoderLayer(
    d_model=d_model,
    num_heads=num_heads,
    d_ff=d_ff,
    dropout=0.0,
)


# --------------------------------------------------
# 1. 기본 Shape
# --------------------------------------------------

batch_size = 2
seq_len = 4

x = torch.randn(
    batch_size,
    seq_len,
    d_model,
)

output, attention_weights = encoder_layer(
    x,
    mask=None,
)

assert output.shape == (
    batch_size,
    seq_len,
    d_model,
)

assert attention_weights.shape == (
    batch_size,
    num_heads,
    seq_len,
    seq_len,
)


# --------------------------------------------------
# 2. 내부 Shape 직접 확인
# --------------------------------------------------

manual_x = x

attention_output, manual_weights = (
    encoder_layer.self_attention(
        manual_x,
        manual_x,
        manual_x,
        None,
    )
)

assert attention_output.shape == (
    batch_size,
    seq_len,
    d_model,
)

attention_output = encoder_layer.dropout1(
    attention_output
)

first_residual = manual_x + attention_output

assert first_residual.shape == (
    batch_size,
    seq_len,
    d_model,
)

manual_x = encoder_layer.norm1(
    first_residual
)

assert manual_x.shape == (
    batch_size,
    seq_len,
    d_model,
)

ffn_output = encoder_layer.feed_forward(
    manual_x
)

assert ffn_output.shape == (
    batch_size,
    seq_len,
    d_model,
)

ffn_output = encoder_layer.dropout2(
    ffn_output
)

second_residual = manual_x + ffn_output

assert second_residual.shape == (
    batch_size,
    seq_len,
    d_model,
)

manual_output = encoder_layer.norm2(
    second_residual
)

assert manual_output.shape == (
    batch_size,
    seq_len,
    d_model,
)


# --------------------------------------------------
# 3. forward와 수동 계산 비교
# --------------------------------------------------

assert torch.allclose(
    output,
    manual_output,
)

assert torch.allclose(
    attention_weights,
    manual_weights,
)


# --------------------------------------------------
# 4. batch / sequence 길이 변경
# --------------------------------------------------

for batch_size, seq_len in [
    (1, 3),
    (3, 5),
    (4, 7),
]:
    x_test = torch.randn(
        batch_size,
        seq_len,
        d_model,
    )

    output_test, weights_test = encoder_layer(
        x_test,
        mask=None,
    )

    assert output_test.shape == (
        batch_size,
        seq_len,
        d_model,
    )

    assert weights_test.shape == (
        batch_size,
        num_heads,
        seq_len,
        seq_len,
    )


print("모든 EncoderLayer 테스트 통과")

모든 EncoderLayer 테스트 통과
